# Verificacion CEN - demanda neta anual

Este notebook procesa los datos horarios nacionales reportados por el Coordinador Electrico Nacional y calcula la demanda neta anual observada en MWh.

Reporte principal:
- Anio
- Registros
- Demanda neta anual observada
- Estado

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

print("[1/10] Librerias cargadas y opciones de visualizacion configuradas.")

[1/10] Librerias cargadas y opciones de visualizacion configuradas.


In [2]:
print("[2/10] Buscando archivo de entrada...")

csv_name = "wp2_elec_input_net_demand_raw.csv"
candidate_paths = [
    Path("../data") / csv_name,
    Path("prototipo_2/data") / csv_name,
    Path.cwd() / "../data" / csv_name,
    Path.cwd() / "prototipo_2/data" / csv_name,
]

data_path = next((path.resolve() for path in candidate_paths if path.exists()), None)

if data_path is None:
    searched_paths = "\n".join(str(path) for path in candidate_paths)
    raise FileNotFoundError(f"No se encontro {csv_name}. Rutas revisadas:\n{searched_paths}")

print(f"Archivo encontrado: {data_path}")

[2/10] Buscando archivo de entrada...
Archivo encontrado: C:\Users\Raimundo Claren\Documents\MERLIN_EDM\prototipo_2\data\wp2_elec_input_net_demand_raw.csv


In [3]:
print("[3/10] Cargando datos horarios y validando columnas requeridas...")

required_columns = {"fecha_hora", "demanda_neta_mwh"}
df = pd.read_csv(data_path)

missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Faltan columnas requeridas: {sorted(missing_columns)}")

df["fecha_hora"] = pd.to_datetime(df["fecha_hora"], errors="coerce")
df["demanda_neta_mwh"] = pd.to_numeric(df["demanda_neta_mwh"], errors="coerce")

invalid_dates = int(df["fecha_hora"].isna().sum())
invalid_demand = int(df["demanda_neta_mwh"].isna().sum())

if invalid_dates:
    raise ValueError(f"Existen {invalid_dates} registros con fecha_hora no parseable.")

if invalid_demand:
    raise ValueError(f"Existen {invalid_demand} registros con demanda_neta_mwh no numerica.")

df = df.sort_values("fecha_hora").reset_index(drop=True)
df["anio"] = df["fecha_hora"].dt.year

print(f"Registros cargados: {len(df):,}")
print(f"Rango temporal: {df['fecha_hora'].min()} a {df['fecha_hora'].max()}")
print(f"Fechas duplicadas: {df['fecha_hora'].duplicated().sum():,}")

[3/10] Cargando datos horarios y validando columnas requeridas...
Registros cargados: 82,104
Rango temporal: 2017-01-01 00:00:00 a 2026-05-18 23:00:00
Fechas duplicadas: 0


In [4]:
print("[4/10] Evaluando completitud temporal por anio...")

def expected_hours_for_year(year: int) -> int:
    year_start = pd.Timestamp(year=year, month=1, day=1)
    return 8784 if year_start.is_leap_year else 8760


def classify_year(row: pd.Series) -> str:
    year = int(row["A\u00f1o"])
    first_expected = pd.Timestamp(year=year, month=1, day=1, hour=0)
    last_expected = pd.Timestamp(year=year, month=12, day=31, hour=23)

    if row["Primera fecha"] > first_expected or row["Ultima fecha"] < last_expected:
        return "Parcial"

    if row["Registros"] == expected_hours_for_year(year) and row["Horas faltantes"] == 0:
        return "Completo"

    return "Incompleto"


missing_hours_by_year = {}
for year, group in df.groupby("anio"):
    full_year_index = pd.date_range(
        start=pd.Timestamp(year=year, month=1, day=1, hour=0),
        end=pd.Timestamp(year=year, month=12, day=31, hour=23),
        freq="h",
    )
    observed_index = pd.DatetimeIndex(group["fecha_hora"])
    missing_hours_by_year[int(year)] = int(len(full_year_index.difference(observed_index)))

completitud = (
    df.groupby("anio", as_index=False)
    .agg(
        Registros=("demanda_neta_mwh", "size"),
        **{
            "Demanda neta anual observada": ("demanda_neta_mwh", "sum"),
            "Primera fecha": ("fecha_hora", "min"),
            "Ultima fecha": ("fecha_hora", "max"),
        },
    )
    .rename(columns={"anio": "A\u00f1o"})
)

completitud["Horas faltantes"] = completitud["A\u00f1o"].map(missing_hours_by_year)
completitud["Estado"] = completitud.apply(classify_year, axis=1)

print("Completitud evaluada.")
print("Resumen de estados:")
print(completitud["Estado"].value_counts().to_string())

[4/10] Evaluando completitud temporal por anio...
Completitud evaluada.
Resumen de estados:
Estado
Completo      8
Incompleto    1
Parcial       1


In [5]:
print("[5/10] Generando reporte anual solicitado...")

reporte_anual = completitud[
    ["A\u00f1o", "Registros", "Demanda neta anual observada", "Estado"]
].copy()

reporte_anual["Demanda neta anual observada"] = reporte_anual[
    "Demanda neta anual observada"
].round(2)

print("Reporte anual listo.")
display(reporte_anual)

[5/10] Generando reporte anual solicitado...
Reporte anual listo.


,Año,Registros,Demanda neta anual observada,Estado
0,2017,8760,"49,315,565.56",Completo
1,2018,8760,"49,363,316.20",Completo
2,2019,8760,"48,791,426.88",Completo
3,2020,8784,"47,422,070.89",Completo
4,2021,8760,"46,782,322.70",Completo
5,2022,8760,"43,170,310.96",Completo
6,2023,8664,"39,748,940.07",Incompleto
7,2024,8784,"38,119,341.63",Completo
8,2025,8760,"37,814,219.81",Completo
9,2026,3312,"13,790,863.47",Parcial


In [6]:
print("Detalle de anios con horas faltantes dentro del anio calendario:")

detalle_faltantes = completitud.loc[
    completitud["Horas faltantes"] > 0,
    ["A\u00f1o", "Horas faltantes", "Primera fecha", "Ultima fecha", "Estado"],
]

display(detalle_faltantes)

Detalle de anios con horas faltantes dentro del anio calendario:


,Año,Horas faltantes,Primera fecha,Ultima fecha,Estado
6,2023,96,2023-01-01,2023-12-31 23:00:00,Incompleto
9,2026,5448,2026-01-01,2026-05-18 23:00:00,Parcial


## Comparacion con balance regional de energia

Esta seccion incorpora el consumo electrico anual por region y sector reportado en el balance regional de energia. Los valores del balance se interpretan como GWh y se convierten a MWh para compararlos con la demanda neta anual observada del CEN.

El valor outlier detectado se mantiene dentro del calculo agregado nacional, pero se reporta explicitamente en una tabla separada.

In [7]:
print("[6/10] Buscando archivo de balance regional de energia...")

sector_csv_name = "wp2_elec_input_sector_shares_raw.csv"
sector_candidate_paths = [
    Path("../data") / sector_csv_name,
    Path("prototipo_2/data") / sector_csv_name,
    Path.cwd() / "../data" / sector_csv_name,
    Path.cwd() / "prototipo_2/data" / sector_csv_name,
]

sector_data_path = next((path.resolve() for path in sector_candidate_paths if path.exists()), None)

if sector_data_path is None:
    searched_paths = "\n".join(str(path) for path in sector_candidate_paths)
    raise FileNotFoundError(f"No se encontro {sector_csv_name}. Rutas revisadas:\n{searched_paths}")

print(f"Archivo encontrado: {sector_data_path}")

[6/10] Buscando archivo de balance regional de energia...
Archivo encontrado: C:\Users\Raimundo Claren\Documents\MERLIN_EDM\prototipo_2\data\wp2_elec_input_sector_shares_raw.csv


In [8]:
print("[7/10] Cargando balance regional y normalizando columnas...")

df_balance = pd.read_csv(sector_data_path)

if df_balance.shape[1] != 4:
    raise ValueError(
        "El archivo de balance regional debe tener 4 columnas: anio, region, sector y valor. "
        f"Columnas encontradas: {list(df_balance.columns)}"
    )

# Se renombran por posicion para evitar problemas de codificacion con anio/region.
df_balance.columns = ["anio", "region", "sector", "valor_gwh"]

df_balance["anio"] = pd.to_numeric(df_balance["anio"], errors="coerce").astype("Int64")
df_balance["region"] = df_balance["region"].astype(str).str.strip()
df_balance["sector"] = df_balance["sector"].astype(str).str.strip()
df_balance["valor_gwh"] = pd.to_numeric(df_balance["valor_gwh"], errors="coerce")

invalid_balance_rows = int(df_balance[["anio", "region", "sector", "valor_gwh"]].isna().any(axis=1).sum())
duplicate_balance_rows = int(df_balance.duplicated(["anio", "region", "sector"]).sum())

if invalid_balance_rows:
    raise ValueError(f"Existen {invalid_balance_rows} registros invalidos en el balance regional.")

if duplicate_balance_rows:
    raise ValueError(f"Existen {duplicate_balance_rows} duplicados anio-region-sector en el balance regional.")

df_balance["anio"] = df_balance["anio"].astype(int)

print(f"Registros cargados: {len(df_balance):,}")
print(f"Anios disponibles: {df_balance['anio'].min()} a {df_balance['anio'].max()}")
print(f"Regiones disponibles: {df_balance['region'].nunique():,}")
print(f"Sectores disponibles: {df_balance['sector'].nunique():,}")

[7/10] Cargando balance regional y normalizando columnas...
Registros cargados: 539
Anios disponibles: 2018 a 2024
Regiones disponibles: 16
Sectores disponibles: 5


In [9]:
print("[8/10] Evaluando cobertura region-sector y detectando outliers...")

expected_regions = sorted(df_balance["region"].unique())
expected_sectors = sorted(df_balance["sector"].unique())
expected_region_sector = pd.MultiIndex.from_product(
    [expected_regions, expected_sectors], names=["region", "sector"]
)

coverage_rows = []
for year, group in df_balance.groupby("anio"):
    observed_region_sector = pd.MultiIndex.from_frame(group[["region", "sector"]].drop_duplicates())
    missing_region_sector = expected_region_sector.difference(observed_region_sector)
    coverage_rows.append(
        {
            "anio": int(year),
            "registros_balance": int(len(group)),
            "regiones_balance": int(group["region"].nunique()),
            "sectores_balance": int(group["sector"].nunique()),
            "combinaciones_region_sector_faltantes": int(len(missing_region_sector)),
        }
    )

cobertura_balance = pd.DataFrame(coverage_rows)
cobertura_balance["Estado balance"] = cobertura_balance[
    "combinaciones_region_sector_faltantes"
].map(lambda value: "Completo" if value == 0 else "Con faltantes tratados como 0")

historical_median = df_balance.groupby(["region", "sector"])["valor_gwh"].transform("median")
df_balance["ratio_vs_mediana_region_sector"] = df_balance["valor_gwh"] / historical_median

outliers_balance = df_balance.loc[
    (historical_median > 0)
    & (df_balance["ratio_vs_mediana_region_sector"] >= 20)
    & (df_balance["valor_gwh"] >= 1000),
    ["anio", "region", "sector", "valor_gwh", "ratio_vs_mediana_region_sector"],
].copy()
outliers_balance["valor_mwh"] = outliers_balance["valor_gwh"] * 1000
outliers_balance = outliers_balance.sort_values("ratio_vs_mediana_region_sector", ascending=False)

print("Cobertura evaluada.")
print("Resumen de estados del balance:")
print(cobertura_balance["Estado balance"].value_counts().to_string())
print(f"Outliers materiales detectados e incorporados en los agregados: {len(outliers_balance):,}")

[8/10] Evaluando cobertura region-sector y detectando outliers...
Cobertura evaluada.
Resumen de estados del balance:
Estado balance
Con faltantes tratados como 0    5
Completo                         2
Outliers materiales detectados e incorporados en los agregados: 0


In [10]:
print("[9/11] Agregando consumo nacional anual del balance regional...")

consumo_balance_anual = (
    df_balance.groupby("anio", as_index=False)
    .agg(
        registros_balance=("valor_gwh", "size"),
        consumo_nacional_anual_balance_gwh=("valor_gwh", "sum"),
    )
    .merge(
        cobertura_balance[
            ["anio", "combinaciones_region_sector_faltantes", "Estado balance"]
        ],
        on="anio",
        how="left",
    )
)

consumo_balance_anual["consumo_nacional_anual_balance_mwh"] = (
    consumo_balance_anual["consumo_nacional_anual_balance_gwh"] * 1000
)

consumo_balance_anual = consumo_balance_anual.sort_values("anio").reset_index(drop=True)

print("Consumo nacional anual agregado. Los valores originales del balance estan en GWh.")
display(consumo_balance_anual)

[9/11] Agregando consumo nacional anual del balance regional...


Consumo nacional anual agregado. Los valores originales del balance estan en GWh.


,anio,registros_balance,consumo_nacional_anual_balance_gwh,combinaciones_region_sector_faltantes,Estado balance,consumo_nacional_anual_balance_mwh
0,2018,75,"73,125.36",5,Con faltantes tratados como 0,"73,125,364.54"
1,2019,75,"74,214.87",5,Con faltantes tratados como 0,"74,214,870.33"
2,2020,76,"74,361.09",4,Con faltantes tratados como 0,"74,361,086.84"
3,2021,75,"76,334.21",5,Con faltantes tratados como 0,"76,334,205.78"
4,2022,78,"80,744.87",2,Con faltantes tratados como 0,"80,744,874.63"
5,2023,80,"82,241.83",0,Completo,"82,241,826.69"
6,2024,80,"82,135.81",0,Completo,"82,135,805.01"


## Consumo nacional Industrial y Transporte del balance regional

Esta seccion suma los consumos regionales del balance para los sectores Industrial y Transporte, reportando el total nacional anual por sector, el total combinado y su comparacion contra la demanda neta agregada del CEN.


In [11]:
print("[10/12] Sumando consumo nacional del balance regional para Industrial y Transporte...")

sectores_objetivo_balance = ["Industrial", "Transporte"]
sectores_objetivo_key = {sector.casefold() for sector in sectores_objetivo_balance}

df_balance_industrial_transporte = df_balance[
    df_balance["sector"].str.casefold().isin(sectores_objetivo_key)
].copy()

sectores_encontrados = set(df_balance_industrial_transporte["sector"].str.casefold().unique())
sectores_faltantes = sorted(sectores_objetivo_key.difference(sectores_encontrados))

if sectores_faltantes:
    raise ValueError(
        "No se encontraron todos los sectores objetivo en el balance regional: "
        f"{sectores_faltantes}"
    )

consumo_balance_industrial_transporte_sector = (
    df_balance_industrial_transporte.groupby(["anio", "sector"], as_index=False)
    .agg(consumo_nacional_sector_gwh=("valor_gwh", "sum"))
    .sort_values(["anio", "sector"])
    .reset_index(drop=True)
)
consumo_balance_industrial_transporte_sector["consumo_nacional_sector_mwh"] = (
    consumo_balance_industrial_transporte_sector["consumo_nacional_sector_gwh"] * 1000
)

consumo_balance_industrial_transporte_total = (
    consumo_balance_industrial_transporte_sector.groupby("anio", as_index=False)
    .agg(
        consumo_industrial_transporte_gwh=("consumo_nacional_sector_gwh", "sum"),
        consumo_industrial_transporte_mwh=("consumo_nacional_sector_mwh", "sum"),
    )
    .sort_values("anio")
    .reset_index(drop=True)
)

print("Consumo nacional anual por sector objetivo:")
display(
    consumo_balance_industrial_transporte_sector.round(
        {"consumo_nacional_sector_gwh": 2, "consumo_nacional_sector_mwh": 2}
    )
)

print("Consumo nacional anual combinado Industrial + Transporte:")
display(
    consumo_balance_industrial_transporte_total.round(
        {"consumo_industrial_transporte_gwh": 2, "consumo_industrial_transporte_mwh": 2}
    )
)

comparacion_cen_industrial_transporte = reporte_anual.rename(
    columns={
        "A\u00f1o": "anio",
        "Demanda neta anual observada": "demanda_neta_anual_cen_mwh",
        "Estado": "Estado CEN",
    }
)[["anio", "demanda_neta_anual_cen_mwh", "Estado CEN"]].merge(
    consumo_balance_industrial_transporte_total,
    on="anio",
    how="inner",
)

comparacion_cen_industrial_transporte["diferencia_industrial_transporte_menos_cen_mwh"] = (
    comparacion_cen_industrial_transporte["consumo_industrial_transporte_mwh"]
    - comparacion_cen_industrial_transporte["demanda_neta_anual_cen_mwh"]
)
comparacion_cen_industrial_transporte["industrial_transporte_sobre_cen_pct"] = (
    comparacion_cen_industrial_transporte["consumo_industrial_transporte_mwh"]
    / comparacion_cen_industrial_transporte["demanda_neta_anual_cen_mwh"]
    * 100
)

comparacion_cen_industrial_transporte = comparacion_cen_industrial_transporte[
    [
        "anio",
        "demanda_neta_anual_cen_mwh",
        "Estado CEN",
        "consumo_industrial_transporte_gwh",
        "consumo_industrial_transporte_mwh",
        "diferencia_industrial_transporte_menos_cen_mwh",
        "industrial_transporte_sobre_cen_pct",
    ]
].round(
    {
        "demanda_neta_anual_cen_mwh": 2,
        "consumo_industrial_transporte_gwh": 2,
        "consumo_industrial_transporte_mwh": 2,
        "diferencia_industrial_transporte_menos_cen_mwh": 2,
        "industrial_transporte_sobre_cen_pct": 2,
    }
)

print("Comparacion Industrial + Transporte del balance regional contra demanda neta anual CEN:")
display(comparacion_cen_industrial_transporte)


[10/12] Sumando consumo nacional del balance regional para Industrial y Transporte...
Consumo nacional anual por sector objetivo:


,anio,sector,consumo_nacional_sector_gwh,consumo_nacional_sector_mwh
0,2018,Industrial,"46,654.01","46,654,014.01"
1,2018,Transporte,"1,229.55","1,229,551.74"
2,2019,Industrial,"46,954.55","46,954,549.62"
3,2019,Transporte,"1,401.25","1,401,254.00"
4,2020,Industrial,"47,328.52","47,328,523.37"
5,2020,Transporte,"1,122.76","1,122,762.08"
6,2021,Industrial,"47,400.00","47,399,996.42"
7,2021,Transporte,"1,244.24","1,244,237.80"
8,2022,Industrial,"50,428.50","50,428,500.06"
9,2022,Transporte,"1,507.70","1,507,702.93"


Consumo nacional anual combinado Industrial + Transporte:


,anio,consumo_industrial_transporte_gwh,consumo_industrial_transporte_mwh
0,2018,"47,883.57","47,883,565.75"
1,2019,"48,355.80","48,355,803.62"
2,2020,"48,451.29","48,451,285.44"
3,2021,"48,644.23","48,644,234.22"
4,2022,"51,936.20","51,936,202.99"
5,2023,"53,377.73","53,377,729.26"
6,2024,"52,508.59","52,508,593.52"


Comparacion Industrial + Transporte del balance regional contra demanda neta anual CEN:


,anio,demanda_neta_anual_cen_mwh,Estado CEN,consumo_industrial_transporte_gwh,consumo_industrial_transporte_mwh,diferencia_industrial_transporte_menos_cen_mwh,industrial_transporte_sobre_cen_pct
0,2018,"49,363,316.20",Completo,"47,883.57","47,883,565.75","-1,479,750.45",97.00
1,2019,"48,791,426.88",Completo,"48,355.80","48,355,803.62","-435,623.26",99.11
2,2020,"47,422,070.89",Completo,"48,451.29","48,451,285.44","1,029,214.55",102.17
3,2021,"46,782,322.70",Completo,"48,644.23","48,644,234.22","1,861,911.52",103.98
4,2022,"43,170,310.96",Completo,"51,936.20","51,936,202.99","8,765,892.03",120.31
5,2023,"39,748,940.07",Incompleto,"53,377.73","53,377,729.26","13,628,789.19",134.29
6,2024,"38,119,341.63",Completo,"52,508.59","52,508,593.52","14,389,251.89",137.75


In [12]:
print("[11/12] Comparando balance regional contra demanda neta anual CEN...")

comparacion_cen_balance = reporte_anual.rename(
    columns={
        "A\u00f1o": "anio",
        "Registros": "registros_cen",
        "Demanda neta anual observada": "demanda_neta_anual_cen_mwh",
        "Estado": "Estado CEN",
    }
).merge(consumo_balance_anual, on="anio", how="inner")

comparacion_cen_balance["diferencia_balance_menos_cen_mwh"] = (
    comparacion_cen_balance["consumo_nacional_anual_balance_mwh"]
    - comparacion_cen_balance["demanda_neta_anual_cen_mwh"]
)
comparacion_cen_balance["diferencia_balance_menos_cen_pct"] = (
    comparacion_cen_balance["diferencia_balance_menos_cen_mwh"]
    / comparacion_cen_balance["demanda_neta_anual_cen_mwh"]
    * 100
)

comparacion_cen_balance = comparacion_cen_balance[
    [
        "anio",
        "registros_cen",
        "demanda_neta_anual_cen_mwh",
        "Estado CEN",
        "registros_balance",
        "consumo_nacional_anual_balance_gwh",
        "consumo_nacional_anual_balance_mwh",
        "Estado balance",
        "diferencia_balance_menos_cen_mwh",
        "diferencia_balance_menos_cen_pct",
    ]
].round(
    {
        "demanda_neta_anual_cen_mwh": 2,
        "consumo_nacional_anual_balance_gwh": 2,
        "consumo_nacional_anual_balance_mwh": 2,
        "diferencia_balance_menos_cen_mwh": 2,
        "diferencia_balance_menos_cen_pct": 2,
    }
)

print("Comparacion anual lista. Anios comunes entre ambas fuentes:")
print(f"{comparacion_cen_balance['anio'].min()} a {comparacion_cen_balance['anio'].max()}")
display(comparacion_cen_balance)

[11/12] Comparando balance regional contra demanda neta anual CEN...
Comparacion anual lista. Anios comunes entre ambas fuentes:
2018 a 2024


,anio,registros_cen,demanda_neta_anual_cen_mwh,Estado CEN,registros_balance,consumo_nacional_anual_balance_gwh,consumo_nacional_anual_balance_mwh,Estado balance,diferencia_balance_menos_cen_mwh,diferencia_balance_menos_cen_pct
0,2018,8760,"49,363,316.20",Completo,75,"73,125.36","73,125,364.54",Con faltantes tratados como 0,"23,762,048.34",48.14
1,2019,8760,"48,791,426.88",Completo,75,"74,214.87","74,214,870.33",Con faltantes tratados como 0,"25,423,443.45",52.11
2,2020,8784,"47,422,070.89",Completo,76,"74,361.09","74,361,086.84",Con faltantes tratados como 0,"26,939,015.95",56.81
3,2021,8760,"46,782,322.70",Completo,75,"76,334.21","76,334,205.78",Con faltantes tratados como 0,"29,551,883.08",63.17
4,2022,8760,"43,170,310.96",Completo,78,"80,744.87","80,744,874.63",Con faltantes tratados como 0,"37,574,563.67",87.04
5,2023,8664,"39,748,940.07",Incompleto,80,"82,241.83","82,241,826.69",Completo,"42,492,886.62",106.90
6,2024,8784,"38,119,341.63",Completo,80,"82,135.81","82,135,805.01",Completo,"44,016,463.38",115.47


## Comparacion anual CEN contra balance regional filtrado al SEN

Esta seccion filtra el balance regional a las regiones pertenecientes al Sistema Electrico Nacional (SEN), agrega su consumo electrico anual y lo compara contra la demanda neta anual agregada reportada por el CEN. La demanda CEN usada en este notebook no tiene desagregacion regional, por lo que la comparacion se realiza contra el agregado anual de las regiones SEN del balance.


In [13]:
print("[12/12] Comparando demanda neta CEN contra balance regional filtrado al SEN...")

regiones_sen = {
    "AP": "XV Region de Arica y Parinacota",
    "TA": "I Region de Tarapaca",
    "AN": "II Region de Antofagasta",
    "AT": "III Region de Atacama",
    "CO": "IV Region de Coquimbo",
    "VS": "V Region de Valparaiso",
    "RM": "Region Metropolitana de Santiago",
    "LI": "VI Region del Libertador General Bernardo O'Higgins",
    "ML": "VII Region del Maule",
    "NB": "XVI Region de Nuble",
    "BI": "VIII Region del Biobio",
    "AR": "IX Region de La Araucania",
    "LR": "XIV Region de Los Rios",
}

regiones_sen_codigos = list(regiones_sen)
regiones_balance_disponibles = set(df_balance["region"].unique())
regiones_sen_faltantes = sorted(set(regiones_sen_codigos).difference(regiones_balance_disponibles))

if regiones_sen_faltantes:
    raise ValueError(
        "Faltan regiones SEN en el balance regional: "
        f"{regiones_sen_faltantes}"
    )

df_balance_sen = df_balance[df_balance["region"].isin(regiones_sen_codigos)].copy()
df_balance_sen["region_nombre"] = df_balance_sen["region"].map(regiones_sen)

expected_sen_region_sector = pd.MultiIndex.from_product(
    [regiones_sen_codigos, expected_sectors], names=["region", "sector"]
)

cobertura_balance_sen_rows = []
for year, group in df_balance_sen.groupby("anio"):
    observed_region_sector = pd.MultiIndex.from_frame(
        group[["region", "sector"]].drop_duplicates()
    )
    missing_region_sector = expected_sen_region_sector.difference(observed_region_sector)
    cobertura_balance_sen_rows.append(
        {
            "anio": int(year),
            "registros_balance_sen": int(len(group)),
            "regiones_sen_balance": int(group["region"].nunique()),
            "sectores_balance_sen": int(group["sector"].nunique()),
            "combinaciones_region_sector_sen_faltantes": int(len(missing_region_sector)),
        }
    )

cobertura_balance_sen = pd.DataFrame(cobertura_balance_sen_rows)
cobertura_balance_sen["Estado balance SEN"] = cobertura_balance_sen[
    "combinaciones_region_sector_sen_faltantes"
].map(lambda value: "Completo" if value == 0 else "Con faltantes tratados como 0")

consumo_balance_sen_anual = (
    df_balance_sen.groupby("anio", as_index=False)
    .agg(
        registros_balance_sen=("valor_gwh", "size"),
        regiones_sen_balance=("region", "nunique"),
        consumo_balance_sen_gwh=("valor_gwh", "sum"),
    )
    .merge(
        cobertura_balance_sen[
            ["anio", "combinaciones_region_sector_sen_faltantes", "Estado balance SEN"]
        ],
        on="anio",
        how="left",
    )
    .sort_values("anio")
    .reset_index(drop=True)
)
consumo_balance_sen_anual["consumo_balance_sen_mwh"] = (
    consumo_balance_sen_anual["consumo_balance_sen_gwh"] * 1000
)

comparacion_cen_balance_sen = reporte_anual.rename(
    columns={
        "A\u00f1o": "anio",
        "Registros": "registros_cen",
        "Demanda neta anual observada": "demanda_neta_anual_cen_mwh",
        "Estado": "Estado CEN",
    }
).merge(consumo_balance_sen_anual, on="anio", how="inner")

comparacion_cen_balance_sen["diferencia_balance_sen_menos_cen_mwh"] = (
    comparacion_cen_balance_sen["consumo_balance_sen_mwh"]
    - comparacion_cen_balance_sen["demanda_neta_anual_cen_mwh"]
)
comparacion_cen_balance_sen["diferencia_balance_sen_menos_cen_pct"] = (
    comparacion_cen_balance_sen["diferencia_balance_sen_menos_cen_mwh"]
    / comparacion_cen_balance_sen["demanda_neta_anual_cen_mwh"]
    * 100
)

comparacion_cen_balance_sen = comparacion_cen_balance_sen[
    [
        "anio",
        "registros_cen",
        "demanda_neta_anual_cen_mwh",
        "Estado CEN",
        "registros_balance_sen",
        "regiones_sen_balance",
        "consumo_balance_sen_gwh",
        "consumo_balance_sen_mwh",
        "Estado balance SEN",
        "diferencia_balance_sen_menos_cen_mwh",
        "diferencia_balance_sen_menos_cen_pct",
    ]
].round(
    {
        "demanda_neta_anual_cen_mwh": 2,
        "consumo_balance_sen_gwh": 2,
        "consumo_balance_sen_mwh": 2,
        "diferencia_balance_sen_menos_cen_mwh": 2,
        "diferencia_balance_sen_menos_cen_pct": 2,
    }
)

print("Regiones SEN consideradas:")
display(
    pd.DataFrame(
        [{"region": code, "region_nombre": name} for code, name in regiones_sen.items()]
    )
)

print("Comparacion anual CEN contra balance regional filtrado a regiones SEN:")
display(comparacion_cen_balance_sen)


[12/12] Comparando demanda neta CEN contra balance regional filtrado al SEN...
Regiones SEN consideradas:


,region,region_nombre
0,AP,XV Region de Arica y Parinacota
1,TA,I Region de Tarapaca
2,AN,II Region de Antofagasta
3,AT,III Region de Atacama
4,CO,IV Region de Coquimbo
5,VS,V Region de Valparaiso
6,RM,Region Metropolitana de Santiago
7,LI,VI Region del Libertador General Bernardo O'Hi...
8,ML,VII Region del Maule
9,NB,XVI Region de Nuble


Comparacion anual CEN contra balance regional filtrado a regiones SEN:


,anio,registros_cen,demanda_neta_anual_cen_mwh,Estado CEN,registros_balance_sen,regiones_sen_balance,consumo_balance_sen_gwh,consumo_balance_sen_mwh,Estado balance SEN,diferencia_balance_sen_menos_cen_mwh,diferencia_balance_sen_menos_cen_pct
0,2018,8760,"49,363,316.20",Completo,61,13,"70,545.28","70,545,279.46",Con faltantes tratados como 0,"21,181,963.26",42.91
1,2019,8760,"48,791,426.88",Completo,62,13,"71,404.98","71,404,984.98",Con faltantes tratados como 0,"22,613,558.10",46.35
2,2020,8784,"47,422,070.89",Completo,62,13,"71,798.04","71,798,038.74",Con faltantes tratados como 0,"24,375,967.85",51.40
3,2021,8760,"46,782,322.70",Completo,62,13,"72,867.54","72,867,538.51",Con faltantes tratados como 0,"26,085,215.81",55.76
4,2022,8760,"43,170,310.96",Completo,64,13,"76,589.59","76,589,588.88",Con faltantes tratados como 0,"33,419,277.92",77.41
5,2023,8664,"39,748,940.07",Incompleto,65,13,"78,608.39","78,608,393.16",Completo,"38,859,453.09",97.76
6,2024,8784,"38,119,341.63",Completo,65,13,"77,171.33","77,171,326.12",Completo,"39,051,984.49",102.45


In [14]:
print("Reporte de outliers del balance regional incorporados en los resultados:")

if outliers_balance.empty:
    print("No se detectaron outliers materiales con la regla ratio >= 20 y valor >= 1.000 GWh.")
else:
    display(outliers_balance.round({"valor_gwh": 2, "valor_mwh": 2, "ratio_vs_mediana_region_sector": 2}))

Reporte de outliers del balance regional incorporados en los resultados:
No se detectaron outliers materiales con la regla ratio >= 20 y valor >= 1.000 GWh.
